<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>WRIT1 Part 2 - Neural Networks</font><br>
<font size=20 color=#00c382>From Scratch MLP - Bank Data</font><br>

|**Assessment Code:** |WRIT1|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S2_25|
|**Course Tutor:**|Imtiaz Hussain Khan, PhD|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**

https://archive.ics.uci.edu/dataset/222/bank+marketing

---

# Initialise Notebook

In [ ]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [ ]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
from matplotlib.pyplot import figure
import matplotlib.ticker as mticker
import matplotlib.animation as animation
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Plotly
import plotly.express as px
import plotly.graph_objects as go

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Pycountry 
import pycountry

# GeoPy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# IPython Display 
from IPython.display import HTML

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [ ]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [ ]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [ ]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [ ]:
internet_required = False

In [ ]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Load and Explore the Dataset

From: https://archive.ics.uci.edu/dataset/222/bank+marketing

The data is related with direct marketing campaigns of a Portuguese banking institution. The marketing campaigns were based on phone calls. Often, more than one contact to the same client was required, in order to access if the product (bank term deposit) would be ('yes') or not ('no') subscribed. 

[This dataset is] bank-full.csv with all examples and 17 inputs, ordered by date (older version of this dataset with less inputs). 

The classification goal is to predict if the client will subscribe (yes/no) a term deposit (variable y).

|Variable Name	|Role	|Type	|Demographic|	Description|	Units|	Missing Values|
|---|---|---|---|---|---|---|
|age|	Feature|	Integer|	Age|	|	|	no|
|job|	Feature	|Categorical	|Occupation	|type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')||		no|
|marital	|Feature|	Categorical|	Marital Status|	marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)||		no|
|education	|Feature|	Categorical	|Education Level|	(categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')||		no|
|default	|Feature|	Binary		||has credit in default?||		no|
|balance	|Feature|	Integer		||average yearly balance|	euros	|no|
|housing	|Feature|	Binary		||has housing loan?||		no|
|loan	|Feature|	Binary	||	has personal loan?||		no|
|contact	|Feature|	Categorical	||	contact communication type (categorical: 'cellular','telephone')||		yes|
|day	|Feature|	Integer	||	last contact day of the month||		unknown|
|month	|Feature|	Date	||	last contact month of the year||		unknown|
|duration	|Feature|	Integer	||	last contact duration, in seconds (numeric)||		unknown|
|campaign	|Feature|	Integer	||	number of contacts performed during this campaign and for this client (numeric, includes last contact)||		unknown|
|pdays	|Feature|	Integer	||number of days that passed by after the client was last contacted from a previous campaign (numeric, -1 means client was not previously contacted)||		unknown|
|previous	|Feature|	Integer	||number of contacts performed before this campaign and for this client (numeric)||		unknown|
|poutcome	|Feature|	Categorical	|| outcome of the previous marketing campaign (categorical: "unknown","other","failure","success")||		unknown|



In [ ]:
if online:
    # Fetch the dataset from the Internet
    df = pd.read_csv('bank-full.csv', delimiter=';')
else:
    # Load the dataset from local file
    df = pd.read_csv('bank-full.csv', delimiter=';')

### Explore the dataset

In [ ]:
print (df.columns)
print ("")
print (df.shape)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')

(45211, 17)


In [ ]:
df.sample(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
3639,52,services,married,secondary,no,108,yes,no,unknown,15,may,543,12,-1,0,unknown,no
20180,31,technician,single,tertiary,no,1463,no,no,cellular,11,aug,97,2,-1,0,unknown,no
42429,64,retired,married,secondary,no,1220,no,no,cellular,7,dec,138,1,-1,0,unknown,no
23253,32,management,single,tertiary,no,0,no,no,cellular,27,aug,121,2,-1,0,unknown,no
5633,35,blue-collar,married,secondary,no,0,yes,no,unknown,23,may,100,4,-1,0,unknown,no
10796,54,blue-collar,single,unknown,no,6622,no,no,unknown,17,jun,83,1,-1,0,unknown,no
4841,36,admin.,married,secondary,no,0,no,no,unknown,21,may,117,1,-1,0,unknown,no
11079,49,technician,married,secondary,no,662,no,no,unknown,18,jun,47,2,-1,0,unknown,no
30018,37,technician,married,secondary,yes,-143,yes,no,cellular,4,feb,95,1,-1,0,unknown,no
22038,34,technician,single,tertiary,no,19,no,no,cellular,20,aug,345,3,-1,0,unknown,no


PCA... feature reduction... not for assessment, but read up.

In [ ]:
df.dropna(inplace=True)
df.shape

(45211, 17)

# Conversion and Encoding Ready for the MLP

In [ ]:
numerical_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
categorical_cols = ['job', 'marital', 'education', 'contact', 'poutcome']

# Binary columns that should not be one-hot encoded
binary_columns = ['default', 'housing', 'loan', 'y']

# Columns to exclude from the system
excluded_features = []

# Split dataset into X (features) and y (labels)
df_X = df.drop('y', axis=1)
df_y = df['y']


In [ ]:
df_X_one_hot = pd.get_dummies(df_X)

In [ ]:
df_X_one_hot.sample(10)

,age,balance,day,duration,campaign,pdays,previous,job_admin.,job_blue-collar,job_entrepreneur,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
11004,56,1126,17,39,3,-1,0,False,False,False,...,True,False,False,False,False,False,False,False,False,True
7113,39,192,29,217,5,-1,0,False,False,False,...,False,False,True,False,False,False,False,False,False,True
12163,42,525,20,10,6,-1,0,False,True,False,...,True,False,False,False,False,False,False,False,False,True
18672,42,1812,31,84,2,-1,0,False,False,False,...,False,False,False,False,False,False,False,False,False,True
27054,34,6107,21,114,3,-1,0,False,False,False,...,False,False,False,True,False,False,False,False,False,True
9897,45,0,9,100,1,-1,0,False,False,False,...,True,False,False,False,False,False,False,False,False,True
24192,53,910,17,156,1,-1,0,False,False,False,...,False,False,False,True,False,False,False,False,False,True
10854,36,214,17,423,1,-1,0,False,True,False,...,True,False,False,False,False,False,False,False,False,True
20397,52,989,11,356,1,-1,0,False,False,False,...,False,False,False,False,False,False,False,False,False,True
40791,55,199,11,151,1,-1,0,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [ ]:
# Reencode the True/False values as floats (True = 1.0, False = 0.0)
df_X_one_hot = df_X_one_hot.astype(float)
df_X_one_hot.sample(10)

,age,balance,day,duration,campaign,pdays,previous,job_admin.,job_blue-collar,job_entrepreneur,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
3165,43.0,123.0,14.0,279.0,2.0,-1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
39554,31.0,138.0,26.0,91.0,1.0,285.0,2.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
33907,21.0,224.0,30.0,871.0,1.0,-1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
42732,56.0,0.0,26.0,413.0,2.0,-1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
24516,56.0,0.0,17.0,194.0,1.0,-1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
23407,40.0,-1545.0,27.0,70.0,13.0,-1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
12902,60.0,468.0,7.0,108.0,2.0,-1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2063,33.0,-257.0,12.0,92.0,4.0,-1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
33942,29.0,3794.0,30.0,249.0,1.0,-1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
33327,50.0,881.0,20.0,162.0,1.0,-1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
df_X_one_hot.dtypes

age                    float64
balance                float64
day                    float64
duration               float64
campaign               float64
pdays                  float64
previous               float64
job_admin.             float64
job_blue-collar        float64
job_entrepreneur       float64
job_housemaid          float64
job_management         float64
job_retired            float64
job_self-employed      float64
job_services           float64
job_student            float64
job_technician         float64
job_unemployed         float64
job_unknown            float64
marital_divorced       float64
marital_married        float64
marital_single         float64
education_primary      float64
education_secondary    float64
education_tertiary     float64
education_unknown      float64
default_no             float64
default_yes            float64
housing_no             float64
housing_yes            float64
loan_no                float64
loan_yes               float64
contact_

### Define Normalisation and Label Encoding Functions

The goal is that all features contribute roughly equally to the training of the neural network.

$ x_scaled = \frac{x - min(x)}{max(x)-min(x)}$



In [ ]:
def max_min_normalize (df, column_name):
    max_value = df[column_name].max()
    min_value = df[column_name].min()
    df[column_name] = (df[column_name] - min_value) / (max_value - min_value)
    return df

# Unused below here... initially?

def label_encode(df, column_name):
    # Get unique values in the column
    unique_values = df[column_name].unique()
    
    # Create a mapping of unique values to integers
    mapping = {value: i for i, value in enumerate(unique_values)}
    
    # Apply the mapping to the column
    df[column_name] = df[column_name].map(mapping).astype(float)
    return df

def encode_binary (df, column_name):
    df[column_name] = df[column_name].map({'yes': 1.0, 'no': 0.0}).astype(float)
    return df


In [ ]:
# Scale each column
for col in df_X_one_hot.columns.to_list():
    df_X_scaled = max_min_normalize(df_X_one_hot, col)

In [ ]:

df_X_scaled.sample(10)

,age,balance,day,duration,campaign,pdays,previous,job_admin.,job_blue-collar,job_entrepreneur,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
34698,0.155844,0.073357,0.133333,0.007320,0.096774,0.190367,0.018182,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
42671,0.597403,0.079776,0.433333,0.075437,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2544,0.376623,0.075527,0.400000,0.020943,0.032258,0.000000,0.000000,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
14120,0.311688,0.072803,0.333333,0.016877,0.016129,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
42470,0.623377,0.095864,0.300000,0.089671,0.016129,0.180046,0.003636,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
23501,0.493506,0.077851,0.900000,0.020740,0.048387,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
31008,0.233766,0.073702,0.300000,0.017487,0.000000,0.098624,0.003636,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
42387,0.181818,0.076344,0.966667,0.030907,0.000000,0.000000,0.000000,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
20821,0.532468,0.075064,0.400000,0.087841,0.064516,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
11559,0.363636,0.074147,0.600000,0.033754,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


Convert the dataframe containing the labels:

In [ ]:
# Convert yes/no outcome to float versions
df_y = df_y.map({'yes': 1.0, 'no': 0.0}).astype(float)

# Reshape for matrix multiplication
y_np=df_y.values.reshape(-1,1)

In [ ]:
X_np = df_X_scaled.to_numpy()

print_output("MLP Input Layer Neurons", df_X_one_hot.shape[1])
print_output("Features shape", X_np.shape)
print_output("Target shape", y_np.shape)


MLP Input Layer Neurons: 51
Features shape: (45211, 51)
Target shape: (45211, 1)


Write out these arrays to files

In [ ]:
# Create filenames, one timestamped, one 'latest'
filepath = "."

X_filename = f'{filepath}/bank_full_X_scaled_{now_date_timestamp}.csv'
X_latest_filename = f'{filepath}/bank_full_X_scaled_latest.csv'

y_filename = f'{filepath}/bank_full_X_scaled_{now_date_timestamp}.csv'
y_latest_filename = f'{filepath}/bank_full_X_scaled_latest.csv'

# Write out CSV files
#df.to_csv(X_filename, index=False)
#df.to_csv(X_latest_filename, index=False)

#df.to_csv(y_filename, index=False)
#df.to_csv(y_latest_filename, index=False)

# Write out numpy arrays

X_np.tofile('data2.csv', sep = ',')
y_np.tofile('data2.csv', sep = ',')

# Neural Network

"Construct a three-layer feedforward neural network consisting of N input neurons, 10 hidden neurons, and M output neuron(s)."

In [ ]:
# Define layer sizes
input_neurons = X_np.shape[1]  # Set dynamically based on the input dataset
hidden_neurons = 10            # 10 hidden neurons, per assessment specification
output_neurons = 1             # This is a binary classification problem, which implies 1 output neuron

# Initialize Weights with small random numbers
# 
#   input_neurons --> weights_1 --> hidden_neurons --> weights_2 --> output_neurons
#                     biases_1                         biases_2
#
np.random.seed(67)

weights_1 = np.random.randn(input_neurons, hidden_neurons) * 0.01
biases_1 = np.zeros((1, hidden_neurons))

weights_2 = np.random.randn(hidden_neurons, output_neurons) * 0.01
biases_2 = np.zeros((1, output_neurons))

Z1​=X⋅W1​+b1​

In a neural network, the forward pass consists of two main operations per layer:

    Linear Transformation: Z=X⋅W+b

    Activation Function: A=σ(Z)

For a binary classification task like this, the Sigmoid function is the standard choice for the output layer because it squashes any input into a value between 0 and 1, which we can interpret as a probability.  (citation)

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    # Derivative of sigmoid is a * (1 - a)
    return a * (1 - a)

# Generic version to allow replacement
def act_func(z):
    return sigmoid(z)

def act_func_deriv(a):
    return sigmoid_derivative(a)

In [ ]:
def forward_propagation(X, W1, b1, W2, b2):
    # Layer 1 (Hidden Layer)
    # Z1 shape: (m, hidden_neurons)
    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid(Z1)
    
    # Layer 2 (Output Layer)
    # Z2 shape: (m, 1)
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)
    
    # We return everything
    return Z1, A1, Z2, A2

To know how "wrong" the network is, we use Binary Cross-Entropy Loss (also called Log Loss). This is much more effective for classification than Mean Squared Error.
Loss=−m1​∑[y⋅log(y^​)+(1−y)⋅log(1−y^​)]

In [ ]:
def compute_loss(y_true, y_pred):
    m = y_true.shape[0]
    # Adding a tiny epsilon (1e-15) prevents log(0) which would result in NaN
    loss = -1/m * np.sum(y_true * np.log(y_pred + 1e-15) + (1 - y_true) * np.log(1 - y_pred + 1e-15))
    return loss

In [ ]:
def back_propagation(X, y, Z1, A1, Z2, A2, W1, W2):
    m = X.shape[0]
    
    # 1. Output Layer Gradients
    dZ2 = A2 - y  # Error signal at the end
    dW2 = (1/m) * np.dot(A1.T, dZ2)
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)
    
    # 2. Hidden Layer Gradients
    # We pass the error back through W2 and then through the Sigmoid derivative
    dZ1 = np.dot(dZ2, W2.T) * sigmoid_derivative(A1)
    dW1 = (1/m) * np.dot(X.T, dZ1)
    db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)
    
    return dW1, db1, dW2, db2

In [ ]:
def train_network(X, y, epochs=1000, learning_rate=0.1):
    # 1. Initialize Weights with the Seed (Reproduction)
    np.random.seed(42)
    input_size = X.shape[1]
    hidden_size = 16
    output_size = 1
    
    W1 = np.random.randn(input_size, hidden_size) * 0.01
    b1 = np.zeros((1, hidden_size))
    W2 = np.random.randn(hidden_size, output_size) * 0.01
    b2 = np.zeros((1, output_size))
    
    loss_history = []

    for i in range(epochs):
        # 2. Forward Propagation
        Z1, A1, Z2, A2 = forward_propagation(X, W1, b1, W2, b2)
        
        # 3. Compute and store Loss
        current_loss = compute_loss(y, A2)
        loss_history.append(current_loss)
        
        # 4. Backpropagation (Calculate Gradients)
        dW1, db1, dW2, db2 = back_propagation(X, y, Z1, A1, Z2, A2, W1, W2)
        
        # 5. Gradient Descent (Update Weights)
        # New Weight = Old Weight - (Learning Rate * Gradient)
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        
        # 6. Print progress every 100 epochs
        if i % 100 == 0:
            print(f"Epoch {i}: Loss {current_loss:.4f}")
            
    return W1, b1, W2, b2, loss_history

In [ ]:
# Start the training!
W1_final, b1_final, W2_final, b2_final, history = train_network(X_np, y_np, epochs=2000, learning_rate=0.5)

# Final Accuracy Check
_, _, _, final_preds = forward_propagation(X_np, W1_final, b1_final, W2_final, b2_final)
final_accuracy = np.mean((final_preds > 0.5) == y_np) * 100
print(f"\nFinal Accuracy: {final_accuracy:.2f}%")

Epoch 0: Loss 0.7014
Epoch 100: Loss 0.3539
Epoch 200: Loss 0.3464
Epoch 300: Loss 0.3382
Epoch 400: Loss 0.3306
Epoch 500: Loss 0.3246
Epoch 600: Loss 0.3199
Epoch 700: Loss 0.3161
Epoch 800: Loss 0.3128
Epoch 900: Loss 0.3098
Epoch 1000: Loss 0.3071
Epoch 1100: Loss 0.3046
Epoch 1200: Loss 0.3023
Epoch 1300: Loss 0.3001
Epoch 1400: Loss 0.2981
Epoch 1500: Loss 0.2962
Epoch 1600: Loss 0.2943
Epoch 1700: Loss 0.2926
Epoch 1800: Loss 0.2909
Epoch 1900: Loss 0.2892

Final Accuracy: 89.34%


In [ ]:
sys.exit('Stopping notebook run here.')   

SystemExit: Stopping notebook run here.

In [ ]:
# 1. Perform the Forward Pass
# Z1, A1 are hidden layer values. 
# Z2, A2  are output layer values
Z1, A1, Z2, y_pred = forward_propagation(X_np, weights_1, biases_1, weights_2, biases_2)

# 2. Calculate the initial Loss
initial_loss = compute_loss(y_np, y_pred)

# 3. Calculate initial Accuracy
# Convert probabilities (0.8, 0.2) into hard predictions (1, 0)
predictions = (y_pred  > 0.5).astype(int)
accuracy = np.mean(predictions == y_np) * 100

print(f"Initial Loss: {initial_loss:.4f}")
print(f"Initial Accuracy: {accuracy:.2f}%")
print(f"Sample Predictions (first 5): \n{y_pred [:5]}")

Initial Loss: 0.6939
Initial Accuracy: 11.70%
Sample Predictions (first 5): 
[[0.50047835]
 [0.50051239]
 [0.50047048]
 [0.50051889]
 [0.50049271]]


---